## pdf multiquery retriever

In [16]:
import os

from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama

os.environ["HTTP_PROXY"] = ''
os.environ["HTTPS_PROXY"] = ''
os.environ["all_proxy"] = ''
os.environ["ALL_PROXY"] = ''

model = ChatOllama(model="qwen2.5:14b", base_url='http://localhost:11434', temperature=0)
embedding_model = OllamaEmbeddings(model='znbang/bge:large-zh-v1.5-f32', base_url='http://localhost:11434')

In [14]:
# Build a sample vectorDB
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load blog post
print('Load blog post')
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
data = loader.load()

# Split
print('Split')
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

# VectorDB
print('vector db')
vectordb = Chroma.from_documents(documents=splits, embedding=embedding_model)

Load blog post
Split
vector db
------------------------
http://localhost:11434/api/embed
------------------------


##  simple use

In [17]:
from langchain.retrievers.multi_query import MultiQueryRetriever

question = "What are the approaches to Task Decomposition?"
# llm = ChatOpenAI(temperature=0)
llm = model
retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(), llm=llm
)

## Set logging for the queries

In [18]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

## 

In [19]:
unique_docs = retriever_from_llm.invoke(question)
len(unique_docs)
# INFO:langchain.retrievers.multi_query:Generated queries: 
# ['What methods can be used for breaking down tasks into smaller components?', 
# 'How can task decomposition be effectively achieved in project management?', 
# 'What techniques are available for dividing complex tasks into simpler sub-tasks?']

------------------------
http://localhost:11434/api/chat
------------------------


INFO:langchain.retrievers.multi_query:Generated queries: ['What methods can be used for breaking down tasks into smaller components?', 'How can task decomposition be effectively achieved in project management?', 'What techniques are available for dividing complex tasks into simpler sub-tasks?']


------------------------
http://localhost:11434/api/embed
------------------------
------------------------
http://localhost:11434/api/embed
------------------------
------------------------
http://localhost:11434/api/embed
------------------------


4

## use your own prompt

In [22]:
zh_prompt = '''你是一名 AI 语言模型助手。
你的任务是生成给定用户问题的五个不同版本，以从向量数据库中检索相关文档。
通过生成用户问题的多个视角，你的目标是帮助用户克服基于距离的相似性搜索的一些限制。
提供这些以换行符分隔的备选问题。
原始问题：{question}'''

In [23]:
from typing import List

from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field


# Output parser will split the LLM result into a list of queries
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Output parser for a list of lines."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))  # Remove empty lines


output_parser = LineListOutputParser()

QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate five 
    different versions of the given user question to retrieve relevant documents from a vector 
    database. By generating multiple perspectives on the user question, your goal is to help
    the user overcome some of the limitations of the distance-based similarity search. 
    Provide these alternative questions separated by newlines.
    Original question: {question}""",
)
llm = model

# Chain
llm_chain = QUERY_PROMPT | llm | output_parser

# Other inputs
question = "What are the approaches to Task Decomposition?"

In [24]:
# Run
retriever = MultiQueryRetriever(
    retriever=vectordb.as_retriever(), llm_chain=llm_chain, parser_key="lines"
)  # "lines" is the key (attribute name) of the parsed output

# Results
unique_docs = retriever.invoke("What does the course say about regression?")
len(unique_docs)

------------------------
http://localhost:11434/api/chat
------------------------


INFO:langchain.retrievers.multi_query:Generated queries: ['What information does the course provide regarding regression analysis?', 'How is regression discussed in the course material?', 'Can you tell me about the coverage of regression in the course?', 'What aspects of regression are covered in the course curriculum?', 'How extensively does the course delve into the topic of regression?']


------------------------
http://localhost:11434/api/embed
------------------------
------------------------
http://localhost:11434/api/embed
------------------------
------------------------
http://localhost:11434/api/embed
------------------------
------------------------
http://localhost:11434/api/embed
------------------------
------------------------
http://localhost:11434/api/embed
------------------------


3

In [25]:
unique_docs

[Document(metadata={'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:', 'language': 'en', 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log"}, page_content='Recency: recent events have higher scores\nImportance: distinguish mundane from core memories. Ask LM directly.\nRelevance: based on how related it is to the current situation / query.\n\n\nReflection mechanism: synthesizes memories into higher level inferences over time and guides the agent’s future behavior. They ar